### Set JAVA_HOME

```shell
export JAVA_HOME=/opt/homebrew/Cellar/openjdk@21/21.0.7/libexec/openjdk.jdk/Contents/Home
```
### Verify JAVA_HOME

```shell
echo $JAVA_HOME
/opt/homebrew/Cellar/openjdk@21/21.0.7/libexec/openjdk.jdk/Contents/Home
```


### Run Opensearch

```shell
cd ~/git/opensearch-3.0.0-SNAPSHOT
./bin/opensearch
```

### Verify if running 

```shell
curl -X GET "http://localhost:9200/" -u admin:admin
```

### View available plugin

```shell
curl -X GET "http://localhost:9200/_cat/plugins?v" -u admin:admin

```


### Install KNN plugin 

```shell
cd ~/git/opensearch-3.0.0-SNAPSHOT
bin/opensearch-plugin install org.opensearch.plugin:opensearch-knn:2.19.0.0
```

- restart is required
  

###   Point to your OpenSearch root installation directory (change this path to where yours is unzipped)
```shell
export OPENSEARCH_HOME="~/git/opensearch-3.0.0-SNAPSHOT"
```

###   Tell the system where the k-NN native libraries are located


```shell
export KNN_LIB_DIR="$OPENSEARCH_HOME/plugins/opensearch-knn/lib"
export DYLD_LIBRARY_PATH="$DYLD_LIBRARY_PATH:$KNN_LIB_DIR"
```

In [16]:
from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Import OllamaEmbeddings for your vectors, and ChatOllama for the final RAG answer stage
from langchain_ollama import OllamaEmbeddings, ChatOllama


In [23]:
# 1. Configure OpenSearch Connection Details
OPENSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "my-hnsw-index"
HTTP_AUTH = ("admin", "admin")  # Update with your credentials if required


In [19]:
# 2. Prepare and chunk your raw text
raw_document_text = """
OpenSearch features a vector database engine capable of handling 
k-NN similarity search, making it an excellent backend choice for 
local, private RAG workflows powered by Ollama.
"""

In [20]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(raw_document_text)
documents = [Document(page_content=chunk, metadata={"source": "local_guide.txt"}) for chunk in chunks]


In [21]:
 
# 3. Define the Ollama Embedding Model
# Ensure you have pulled this model locally via terminal first: `ollama pull nomic-embed-text`
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

### Index creation 
If index does not exists create it 
```
curl -X PUT "http://localhost:9200/my-ollama-rag-index" \
     -H 'Content-Type: application/json' \
     -u admin:admin
```

### View Index 
```
{"acknowledged":true,"shards_acknowledged":true,"index":"my-ollama-rag-index"}
```
View the index using 
```
curl -X GET "http://localhost:9200/_cat/indices?v"
```
you should get a response : 


```
health status index               uuid                   pri rep docs.count docs.deleted store.size pri.store.size
yellow open   my-ollama-rag-index W_KrY3CnQDi5zWAdQoZ7IA   1   1          0            0       208b           208b
```


### Index Creation attempt 2 : 

```shell
curl -X PUT "http://localhost:9200/my-hnsw-index" \
     -H 'Content-Type: application/json' \
     -u admin:admin \
     -d '{
  "settings": {
    "index.knn": true
  },
  "mappings": {
    "properties": {
      "my_vector": {
        "type": "knn_vector",
        "dimension": 128,
        "method": {
          "name": "hnsw",
          "space_type": "l2",
          "engine": "faiss",
          "parameters": {
            "ef_construction": 128,
            "m": 24
          }
        }
      }
    }
  }
}'

```

In [24]:
# 4. Insert documents and embeddings into OpenSearch
docsearch = OpenSearchVectorSearch.from_documents(
    documents=documents,
    embedding=embedding_model,
    opensearch_url=OPENSEARCH_URL,
    index_name=INDEX_NAME,
    http_auth=HTTP_AUTH,
    use_ssl=False,
    verify_certs=False,
    engine="nmslib",
    space_type="cosinesimil"
)


### View Documents in the index 

```
curl -X GET "http://localhost:9200/my-hnsw-index/_search?pretty" \
     -H 'Content-Type: application/json' \
     -d '{
       "size": 100,
       "query": {
         "match_all": {}
       }
     }'
```

### Ingestion Complete

In [ ]:
### Semantic Search 

query = "How does vector search work?"

# Retrieve top 2 closest documents
results = docsearch.similarity_search(query, k=2)

for doc in results:
    print(f"\nContent: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")


In [9]:
# Force-create the index correctly using .from_documents()
docsearch = OpenSearchVectorSearch.from_documents(
    documents=documents,
    embedding=embedding_model,
    opensearch_url=OPENSEARCH_URL,
    index_name="my-ollama-rag-index",
    # THESE FOUR LINES ENFORCE THE KNN ENGINE:
    engine="nmslib",              
    space_type="cosinesimil",     
    index_params={"parameters": {"engine": "nmslib", "method": "hnsw"}},
    http_auth=HTTP_AUTH
)


In [ ]:

 




print(f"Successfully inserted documents into OpenSearch using Ollama embeddings!")

# 5. Optional: How you would use ChatOllama later to query it (RAG)
# llm = ChatOllama(model="llama3") 


In [27]:
docsearch = OpenSearchVectorSearch(
    opensearch_url="http://localhost:9200",
    index_name="my-hnsw-index",
    embedding_function=embedding_model,  # Your embedding model instance
    http_auth=("admin", "admin"),
    use_ssl=False,
    verify_certs=False,
    
    # MATCHING YOUR EXACT INDEX SCHEMA:
    vector_field="vector_field",
    text_field="text"
)

In [ ]:
results = docsearch.similarity_search("does opensearch support knn", k=1)
for doc in results:
    print(doc.page_content)

### Delete Index 

```shell 
curl -X DELETE "http://localhost:9200/my-hnsw-index" -u admin:admin
```

In [33]:
docsearch = OpenSearchVectorSearch(
    opensearch_url="http://localhost:9200",
    index_name="my-hnsw-index",
    embedding_function=embedding_model,  # Your 128-dim embedding instance
    http_auth=("admin", "admin"),
    use_ssl=False,
    verify_certs=False,
    vector_field="vector_field",
    text_field="text"
)

In [ ]:
docsearch.add_documents(documents=documents)
